# RAG avanzado

En el notebook anterior construimos un RAG **a mano** con `numpy`: nosotros calculábamos los embeddings, los guardábamos en una variable y programábamos la búsqueda. Perfecto para **entender** cómo funciona.

Pero en un proyecto real eso no escala: con miles de documentos necesitas algo más serio. Ahí entran las **bases de datos vectoriales**. Hoy usaremos **ChromaDB**, que es la opción **más simple y a la vez profesional**:

- Calcula los **embeddings por ti** (no hace falta sentence-transformers a mano).
- **Guarda** los documentos y sus vectores, y **persiste en disco** (sobreviven al reinicio).
- Hace la **búsqueda por significado** optimizada.
- Permite **filtrar por metadatos** (por fecha, sección, autor…).

### Antes (a mano) vs ahora (con Chroma)

| Tarea | Notebook anterior (a mano) | Ahora (ChromaDB) |
|-------|----------------------------|------------------|
| Crear embeddings | `sentence-transformers` a mano | **Automático** |
| Guardarlos | una variable de Python | **Base de datos en disco** |
| Buscar | `numpy` + `argsort` | `collection.query(...)` |
| Filtrar / metadatos | habría que programarlo | **Incluido** |


## Configuración

Instalamos **ChromaDB** (incluye todo lo necesario, hasta el modelo de embeddings) y **openai** para la generación.

In [19]:
!pip install -q chromadb

In [20]:
from openai import OpenAI
from getpass import getpass

API_KEY = getpass("Pega tu clave de Groq u OpenRouter: ")


In [47]:
BASE_URL = "https://api.groq.com/openai/v1"
MODELO   = "meta-llama/llama-4-scout-17b-16e-instruct"

cliente = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [22]:
import json

## Creamos la base de datos vectorial

Usamos un **cliente persistente**: guarda los datos en una carpeta (`./chroma_data`), así no hay que reindexar cada vez. Dentro creamos una **colección** (como una "tabla").

> La primera vez, Chroma descarga un pequeño modelo de embeddings. Tarda un momento solo la primera vez.

In [23]:
import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(path="./bbdd_vector")
coleccion = chroma_client.get_or_create_collection(name="documentos_locales")

## Añadimos nuestros documentos (con metadatos)

Igual que antes, usamos la documentación. La novedad: a cada documento le ponemos un **id** y unos **metadatos** (aquí, la `seccion`), que luego servirán para **citar** y **filtrar**.

Lo importante: **Chroma calcula los embeddings automáticamente** al añadir. No tenemos que hacer nada más.

In [34]:
import os
from pypdf import PdfReader

def cargar_documentos():
    """
    Lee todos los archivos PDF y TXT de la carpeta local, los fragmenta 
    en bloques legibles y los guarda en ChromaDB.
    """
    documentos_para_guardar = []
    ids_para_guardar = []
    
    # Tamaño del fragmento (chunk) en caracteres para no desbordar al LLM
    CHUNK_SIZE = 1000 
    
    for archivo in os.listdir("./bbdd_vector"):
        ruta_completa = os.path.join("./bbdd_vector", archivo)
        texto_completo = ""
        
        # Caso A: Si es un archivo PDF
        if archivo.endswith(".pdf"):
            try:
                reader = PdfReader(ruta_completa)
                for numero_pagina, pagina in enumerate(reader.pages):
                    texto_completo += pagina.extract_text() or ""
            except Exception as e:
                print(f"Error leyendo el PDF {archivo}: {e}")
                continue
                
        # Caso B: Si sigue siendo un archivo TXT estándar
        elif archivo.endswith(".txt"):
            with open(ruta_completa, "r", encoding="utf-8") as f:
                texto_completo = f.read()
        else:
            continue # Ignorar otros formatos de archivo (imágenes, exe, etc.)

        if not texto_completo.strip():
            continue

        # FRAGMENTACIÓN (Chunking): Dividimos el texto largo en fragmentos más pequeños
        for i in range(0, len(texto_completo), CHUNK_SIZE):
            fragmento = texto_completo[i : i + CHUNK_SIZE]
            documentos_para_guardar.append(fragmento)
            # El ID debe ser único por fragmento: "nombre_archivo_parte_0", "nombre_archivo_parte_1"...
            ids_para_guardar.append(f"{archivo}_parte_{i}")

    if documentos_para_guardar:
        coleccion.add(documents=documentos_para_guardar, ids=ids_para_guardar)
        print(f"Éxito: Se han indexado {len(documentos_para_guardar)} fragmentos extraídos de tus PDFs/TXTs.")
    else:
        print("Carpeta vacía o sin archivos PDF/TXT válidos.")

In [35]:
cargar_documentos()

Éxito: Se han indexado 15 fragmentos extraídos de tus PDFs/TXTs.


## Función para llamar los datos. (tools)

In [42]:
def buscar_documentos(termino_busqueda: str) -> str:
    """
    Función de Python que consulta la base de datos vectorial local.
    """
    resultados = coleccion.query(query_texts=[termino_busqueda], n_results=1)
    
    if not resultados['documents'][0]:
        return "No se encontró información relevante en los documentos locales."
        
    contexto = resultados['documents'][0][0]
    return f"[Resultado]: {contexto}"

In [43]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "buscar_en_documentos",
            "description": "Busca información privada, manuales de soporte o datos internos de la escuela en la base de datos de archivos locales.",
            "parameters": {
                "type": "object",
                "properties": {
                    "termino_busqueda": {
                        "type": "string",
                        "description": "La palabra clave o frase que se quiere buscar en los documentos.",
                    }
                },
                "required": ["termino_busqueda"],
            },
        },
    }
]

funciones = {
    "buscar_en_documentos": buscar_documentos
}

## Función llamar agente.

In [45]:
def ejecutar_agente(mensajes, tools, funciones, max_pasos=15, verbose=True):
    """Bucle de function calling: el modelo pide herramientas, nosotros las ejecutamos."""
    for paso in range(1, max_pasos + 1):
        respuesta = cliente.chat.completions.create(
            model=MODELO,
            messages=mensajes,
            tools=tools,            
            tool_choice="auto",     
            temperature=0,
        )
        msg = respuesta.choices[0].message

        if not msg.tool_calls:
            return msg.content

        # Registramos la intención del asistente de llamar a la herramienta
        mensajes.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        # Ejecutamos las herramientas solicitadas por el modelo
        for tc in msg.tool_calls:
            nombre = tc.function.name
            argumentos = json.loads(tc.function.arguments or "{}")
            funcion = funciones.get(nombre)
            if funcion is None:
                resultado = f"Error: la herramienta '{nombre}' no existe."
            else:
                try:
                    resultado = funcion(**argumentos)
                except Exception as e:
                    resultado = f"Error al ejecutar {nombre}: {e}"
            if verbose:
                print(f"[Paso {paso}] {nombre}({argumentos}) → {str(resultado)[:120]}")
            
            # Le devolvemos el contenido a OpenAI con el rol específico "tool"
            mensajes.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "name": nombre,
                "content": str(resultado),
            })

    return "He alcanzado el límite de pasos."

## Ejemplo.

In [48]:

historial_mensajes = [
    {
        "role": "system", 
        "content": "Eres un asistente de un bootcamp de Data Science"
    },
    {
        "role": "user", 
        "content": "¿Que proyectos hay en el bootcamp?"
    }
]

print("Iniciando ejecución del agente...")
respuesta_final = ejecutar_agente(historial_mensajes, tools, funciones)
print(f"\n[Respuesta Final]: {respuesta_final}")

Iniciando ejecución del agente...
[Paso 1] buscar_en_documentos({'termino_busqueda': 'proyectos bootcamp'}) → [Resultado]:  de ML que utiliza
redes neuronales con varias 
capas.Hablemos de aplicaciones IHablemos de aplicaciones II

[Respuesta Final]: En el Bootcamp de Data Science, hay varios proyectos que se trabajan a lo largo del curso. Algunos de los proyectos mencionados son:

- Proyecto de Machine Learning 
- Proyecto de Data Analysis 
- Proyecto de Data Engineering 

Estos proyectos tienen como objetivo afianzar los fundamentos del lenguaje Python, construir modelos de aprendizaje automático de datos, conocer cómo la ciencia de datos se convierte en un proceso de negocio y desplegar soluciones de forma escalable. 

También se trabajan proyectos relacionados con:

- Modelos supervisados
- Modelos no supervisados
- Deep Learning
- Inteligencia Artificial

Estos proyectos se trabajan en diferentes módulos del curso, como:

- Fundamentals
- Data Analytics
- Data Engineering
- Machin